# MW Scan Analysis

Run cells top-to-bottom. Cell 1 lists recent runs, Cell 2 sets which RID to analyze, Cell 3+ loads, plots, and fits.

In [ ]:
# Setup (run once)
import sys, os
sys.path.insert(0, r"C:\Networking Experiment\Claude_TEST_faisal\useful_codes\repository")

import numpy as np
import matplotlib.pyplot as plt
import h5py
import glob
import datetime
from pathlib import Path

from analysis.data_loader import load_h5, find_runs, RESULTS_DIR
from analysis.microwave_analysis import fit_scan_result

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print(f"Results directory: {RESULTS_DIR}")
print("Setup complete.")

## 1. List Recent Runs
Re-run this cell to refresh (e.g. after a new experiment finishes).

In [ ]:
files = find_runs(format="h5")
if not files:
    print(f"No HDF5 files in {RESULTS_DIR}")
else:
    rows = []
    for f in files[-30:]:
        fname = os.path.basename(f)
        parts = fname.replace(".h5", "").split("-", 1)
        rid = parts[0] if parts else "?"
        exp = parts[1] if len(parts) > 1 else "?"
        folder = str(Path(f).parent.relative_to(RESULTS_DIR))
        dt = datetime.datetime.fromtimestamp(os.path.getmtime(f))
        mode = typ = ""
        try:
            with h5py.File(f, "r") as hf:
                ds = hf.get("datasets")
                if ds is not None:
                    for key in ["mw_scan_mode", "mw_scan_type"]:
                        if key in ds:
                            val = ds[key][()]
                            val = val.decode() if isinstance(val, bytes) else str(val)
                        elif key in ds.attrs:
                            val = ds.attrs[key]
                            val = val.decode() if isinstance(val, bytes) else str(val)
                        else:
                            val = ""
                        if key == "mw_scan_mode": mode = val
                        else: typ = val
        except Exception:
            pass
        rows.append((rid, exp, mode, typ, dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M"), folder))

    print(f"{'RID':>9s}  {'Experiment':<22s}  {'Mode':<12s}  {'Type':<11s}  {'Date':<12s}  {'Time':<6s}  Folder")
    print(f"{'-'*9}  {'-'*22}  {'-'*12}  {'-'*11}  {'-'*12}  {'-'*6}  {'-'*20}")
    for r in reversed(rows):
        print(f"{r[0]:>9s}  {r[1]:<22s}  {r[2]:<12s}  {r[3]:<11s}  {r[4]:<12s}  {r[5]:<6s}  {r[6]}")

## 2. Select Run
Set `RID` below and run the cell. Use `None` for the latest run.

In [ ]:
########################################
RID = None   # <-- set RID here, or None for latest
########################################

if RID is not None:
    pattern = f"{RID:09d}-*.h5"
    matches = glob.glob(str(RESULTS_DIR / "**" / pattern), recursive=True)
    if not matches:
        raise FileNotFoundError(f"No HDF5 for RID {RID} in {RESULTS_DIR}")
    filepath = matches[0]
else:
    all_files = find_runs(format="h5")
    if not all_files:
        raise FileNotFoundError(f"No HDF5 files in {RESULTS_DIR}")
    filepath = all_files[-1]
    print("(Using latest run)")

print(f"File: {os.path.basename(filepath)}")
print(f"Path: {filepath}")

result = load_h5(filepath)
data = result["data"]
meta = result["metadata"]
archive = result["archive"]

## 3. Summary & Raw Data Table

In [ ]:
for k, v in meta.items():
    val = v.decode() if isinstance(v, bytes) else v
    print(f"  {k}: {val}")

mode = data.get("mw_scan_mode", b"?")
if isinstance(mode, bytes): mode = mode.decode()
stype = data.get("mw_scan_type", b"?")
if isinstance(stype, bytes): stype = stype.decode()

if "mw_scan_x" in data:
    x_raw = np.asarray(data["mw_scan_x"], dtype=float)
    unique_x = np.unique(x_raw)
    n_pts = len(unique_x)
    n_total = len(x_raw)
    n_loops = n_total // max(n_pts, 1)
    print(f"\n  Mode: {mode}  |  Type: {stype}")
    print(f"  Scan points: {n_pts}  |  Loops: {n_loops}  |  Total: {n_total}")
    if stype == "frequency":
        print(f"  Range: {unique_x[0]/1e6:.4f} .. {unique_x[-1]/1e6:.4f} MHz")
    else:
        print(f"  Range: {unique_x[0]:.0f} .. {unique_x[-1]:.0f} mu")

    def avg_by_x(key):
        if key not in data: return None
        arr = np.asarray(data[key], dtype=float)
        return np.array([np.mean(arr[x_raw == xv]) for xv in unique_x]) if arr.size == x_raw.size else None

    # Retention = N(shot1>thresh AND shot2>thresh) / N(shot1>thresh)
    # Uses mw_atom_loaded and mw_atom_retained directly (works for old and new data)
    loaded_raw = np.asarray(data.get("mw_atom_loaded", []), dtype=float)
    retained_raw = np.asarray(data.get("mw_atom_retained", []), dtype=float)
    load_rate = avg_by_x("mw_atom_loaded")
    n_per = np.array([np.sum(x_raw == xv) for xv in unique_x])
    ret = np.zeros(len(unique_x))
    err = np.zeros(len(unique_x))
    n_loaded_per = np.zeros(len(unique_x), dtype=int)
    for i, xv in enumerate(unique_x):
        mask = x_raw == xv
        l = loaded_raw[mask] if loaded_raw.size == x_raw.size else np.zeros(mask.sum())
        r = retained_raw[mask] if retained_raw.size == x_raw.size else np.zeros(mask.sum())
        n_loaded = int(l.sum())
        n_retained = int((l * r).sum())
        n_loaded_per[i] = n_loaded
        if n_loaded > 0:
            ret[i] = n_retained / n_loaded
            err[i] = np.sqrt(max(ret[i]*(1-ret[i]), 0) / n_loaded)
        else:
            ret[i] = float('nan')
            err[i] = float('nan')

    s1 = avg_by_x("mw_shot1_spcm1"); s2 = avg_by_x("mw_shot1_spcm2")
    p1 = avg_by_x("mw_shot2_spcm1"); p2 = avg_by_x("mw_shot2_spcm2")
    b1 = avg_by_x("mw_bg_spcm1");    b2 = avg_by_x("mw_bg_spcm2")
    shot1_tot = (s1 + s2) if s1 is not None and s2 is not None else None
    shot2_tot = (p1 + p2) if p1 is not None and p2 is not None else None
    bg_tot = (b1 + b2) if b1 is not None and b2 is not None else None

    x_label = "Freq (MHz)" if stype == "frequency" else "Time (mu)"
    x_fmt = [f"{v/1e6:.4f}" for v in unique_x] if stype == "frequency" else [f"{v:.0f}" for v in unique_x]

    print(f"\n  {x_label:<14s}  {'N':>4s}  {'Nlod':>4s}  {'Ret':>6s}  {'Err':>6s}  {'Load':>5s}  {'Shot1':>6s}  {'Shot2':>6s}  {'BG':>5s}")
    print(f"  {'-'*14}  {'-'*4}  {'-'*4}  {'-'*6}  {'-'*6}  {'-'*5}  {'-'*6}  {'-'*6}  {'-'*5}")
    for i in range(len(unique_x)):
        row = f"  {x_fmt[i]:<14s}  {int(n_per[i]):>4d}  {n_loaded_per[i]:>4d}"
        row += f"  {ret[i]:>6.3f}" if not np.isnan(ret[i]) else "     --"
        row += f"  {err[i]:>6.3f}" if not np.isnan(err[i]) else "     --"
        row += f"  {load_rate[i]:>5.2f}" if load_rate is not None else "      "
        row += f"  {shot1_tot[i]:>6.1f}" if shot1_tot is not None else "       "
        row += f"  {shot2_tot[i]:>6.1f}" if shot2_tot is not None else "       "
        row += f"  {bg_tot[i]:>5.1f}" if bg_tot is not None else "      "
        print(row)
else:
    print("\nNo MW scan data in this file.")
    print("Datasets:", sorted(data.keys()))

## 4. Plot Retention + Counts
Re-run this cell to refresh after more data is collected (incremental saves).

In [ ]:
# Reload data (picks up incremental saves)
result = load_h5(filepath)
data = result["data"]

x_raw = np.asarray(data["mw_scan_x"], dtype=float)
unique_x = np.unique(x_raw)
loaded_raw = np.asarray(data.get("mw_atom_loaded", []), dtype=float)
retained_raw = np.asarray(data.get("mw_atom_retained", []), dtype=float)

def avg_by_x(key):
    if key not in data: return None
    arr = np.asarray(data[key], dtype=float)
    return np.array([np.mean(arr[x_raw == xv]) for xv in unique_x]) if arr.size == x_raw.size else None

stype = data.get("mw_scan_type", b"frequency")
if isinstance(stype, bytes): stype = stype.decode()
mode = data.get("mw_scan_mode", b"MW Scan")
if isinstance(mode, bytes): mode = mode.decode()

if stype == "frequency":
    xp = unique_x / 1e6
    xlabel = "Frequency (MHz)"
else:
    xp = unique_x
    xlabel = "Duration (mu)" if stype == "time" else "Free evolution (mu)"

# Retention = N(shot1>thresh AND shot2>thresh) / N(shot1>thresh)
ret = np.zeros(len(unique_x))
err = np.zeros(len(unique_x))
for i, xv in enumerate(unique_x):
    mask = x_raw == xv
    l = loaded_raw[mask] if loaded_raw.size == x_raw.size else np.zeros(mask.sum())
    r = retained_raw[mask] if retained_raw.size == x_raw.size else np.zeros(mask.sum())
    n_loaded = int(l.sum())
    n_retained = int((l * r).sum())
    if n_loaded > 0:
        ret[i] = n_retained / n_loaded
        err[i] = np.sqrt(max(ret[i]*(1-ret[i]), 0) / n_loaded)
    else:
        ret[i] = float('nan')
        err[i] = float('nan')
load_rate = avg_by_x("mw_atom_loaded")

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True,
                         gridspec_kw={"height_ratios": [3, 2, 1.5]})

ax1 = axes[0]
valid_mask = ~np.isnan(ret)
if valid_mask.any():
    ax1.errorbar(xp[valid_mask], ret[valid_mask], yerr=err[valid_mask],
                 fmt="o", color="C0", capsize=3, markersize=6, label="Retention")
ax1.set_ylabel("Retention = N(s1>t & s2>t) / N(s1>t)")
ax1.set_ylim(-0.05, 1.05)
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)
ax1.set_title(f"{mode} (RID {data.get('rid', '?')})")

ax2 = axes[1]
s1 = avg_by_x("mw_shot1_spcm1"); s2 = avg_by_x("mw_shot1_spcm2")
p1 = avg_by_x("mw_shot2_spcm1"); p2 = avg_by_x("mw_shot2_spcm2")
b1 = avg_by_x("mw_bg_spcm1");    b2 = avg_by_x("mw_bg_spcm2")
if s1 is not None and s2 is not None:
    ax2.plot(xp, s1+s2, "s-", color="C2", ms=4, label="Shot1 (loaded)")
if p1 is not None and p2 is not None:
    ax2.plot(xp, p1+p2, "D-", color="C4", ms=4, label="Shot2 (post-MW)")
if b1 is not None and b2 is not None:
    ax2.plot(xp, b1+b2, "x-", color="C7", ms=4, label="Background")
ax2.set_ylabel("Avg photon counts")
ax2.legend(fontsize=8, loc="upper right")
ax2.grid(True, alpha=0.3)

ax3 = axes[2]
if load_rate is not None:
    ax3.plot(xp, load_rate, "o-", color="C1", ms=4)
    ax3.axhline(1.0, color="gray", ls="--", lw=0.5)
ax3.set_ylabel("Loading rate")
ax3.set_ylim(-0.05, 1.15)
ax3.set_xlabel(xlabel)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Total: {len(x_raw)} shots, {len(x_raw)//max(len(unique_x),1)} loops")

## 5. Fit
Gaussian dip (frequency) / Rabi (time) / Ramsey fit with overlay.

In [ ]:
########################################
FIT_MIN = None   # e.g. 249.9 (MHz for freq, mu for time). None = no lower bound
FIT_MAX = None   # e.g. 250.1. None = no upper bound
########################################

fit_result = None
fitter = None
valid = ~np.isnan(ret)

# Apply fit range restriction
if FIT_MIN is not None or FIT_MAX is not None:
    x_check = unique_x / 1e6 if stype == "frequency" else unique_x
    if FIT_MIN is not None:
        valid = valid & (x_check >= FIT_MIN)
    if FIT_MAX is not None:
        valid = valid & (x_check <= FIT_MAX)
    print(f"Fit range: [{FIT_MIN}, {FIT_MAX}] -> {valid.sum()} points")

if valid.sum() >= 3:
    x_fit = unique_x[valid]
    ret_fit = ret[valid]
    err_fit = err[valid].copy()
    err_fit[err_fit == 0] = 0.01

    # Debug: show what goes into fitter
    print(f"Fitting {valid.sum()} points:")
    for j in range(len(x_fit)):
        xv = x_fit[j] / 1e6 if stype == "frequency" else x_fit[j]
        print(f"  x={xv:.4f}  ret={ret_fit[j]:.3f}  err={err_fit[j]:.3f}")

    if stype == "frequency":
        from fitting.gaussian_dip import GaussianDipFit
        fitter = GaussianDipFit()
        x_mhz = x_fit / 1e6
        fit_res = fitter.fit_lsq(x_mhz, ret_fit, yerr=err_fit)
        if fit_res["success"]:
            center, center_err = fitter.center(fit_res)
            fwhm, fwhm_err = fitter.fwhm(fit_res)
            print(f"\nFIT SUCCEEDED")
            print(f"  Resonance: {center:.6f} +/- {center_err:.6f} MHz")
            print(f"  FWHM: {fwhm:.6f} +/- {fwhm_err:.6f} MHz")
            fit_result = fit_res
        else:
            print("\nFIT FAILED (see error above)")
    elif stype == "time":
        from fitting.damped_cosine import RabiFit
        fitter = RabiFit()
        fit_res = fitter.fit_lsq(x_fit, ret_fit, yerr=err_fit)
        if fit_res["success"]:
            pi_t = fitter.pi_time(fit_res)
            print(f"\nFIT SUCCEEDED\n  Pi time: {pi_t:.0f} mu")
            fit_result = fit_res
        else:
            print("\nFIT FAILED (see error above)")
    else:
        print(f"No fitter for type '{stype}'")
else:
    print(f"Cannot fit: {valid.sum()} valid points (need >= 3)")

# Plot ALL data + fit overlay
fig, ax = plt.subplots(figsize=(10, 5))
all_valid = ~np.isnan(ret)
if all_valid.any():
    ax.errorbar(xp[all_valid], ret[all_valid], yerr=err[all_valid], fmt="o", color="C0",
                capsize=3, ms=6, label="Data")
if FIT_MIN is not None:
    ax.axvspan(xp.min() - 0.01, FIT_MIN, alpha=0.1, color="gray")
if FIT_MAX is not None:
    ax.axvspan(FIT_MAX, xp.max() + 0.01, alpha=0.1, color="gray")

if fit_result and fit_result.get("success") and fitter is not None:
    x_fine = np.linspace(unique_x[valid].min(), unique_x[valid].max(), 500)
    x_fine_plot = x_fine / 1e6 if stype == "frequency" else x_fine
    x_fine_fit = x_fine / 1e6 if stype == "frequency" else x_fine
    y_fine = fitter.evaluate(x_fine_fit, fit_result["params"])
    ax.plot(x_fine_plot, y_fine, "r-", lw=2, alpha=0.8, label="Fit")
else:
    ax.text(0.98, 0.02, "Fit failed or insufficient data", transform=ax.transAxes,
            fontsize=9, va="bottom", ha="right", color="gray", fontstyle="italic")

ax.set_xlabel(xlabel)
ax.set_ylabel("Retention P(retained | loaded)")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"{mode} (RID {data.get('rid', '?')}) - Fit")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Browse All Datasets

In [ ]:
print("=== datasets/ ===")
for k in sorted(data.keys()):
    v = data[k]
    if isinstance(v, np.ndarray) and v.size > 0:
        print(f"  {k:40s}  {str(v.dtype):8s}  shape={str(v.shape):12s}  [{v.min():.4g} .. {v.max():.4g}]")
    elif isinstance(v, np.ndarray):
        print(f"  {k:40s}  {str(v.dtype):8s}  shape={str(v.shape):12s}  (empty)")
    else:
        val = v.decode() if isinstance(v, bytes) else v
        print(f"  {k:40s}  = {val}")

print(f"\n=== archive/ ({len(archive)} keys) ===")
for k in sorted(archive.keys())[:20]:
    v = archive[k]
    val = v.decode() if isinstance(v, bytes) else v
    print(f"  {k:40s}  = {val}")
if len(archive) > 20:
    print(f"  ... and {len(archive)-20} more")